# Entrenar el checkpoint NLLB-200 + LoRA (F. Prado) para retrotraducción

Este checkpoint es lo que necesita `4_aumento_datos/retrotraduccion.py` para traducir
las paráfrasis en español al shiwilu (paso 2 de la técnica).

Antes de correr: `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> **GPU**.

**IMPORTANTE:** cada celda de abajo empieza con `%cd /content/...` explícito, a
propósito — Colab a veces reinicia el entorno solo (por ejemplo tras instalar
`torch`), y eso borra en qué carpeta estabas parada. Si ves un error de
`ModuleNotFoundError` o `No such file or directory`, casi siempre es por eso:
vuelve a correr **desde la celda 1** en adelante, no solo la que falló.

## 1. Montar Google Drive (para guardar el checkpoint antes de que se cierre la sesión)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonar tu repo y el de F. Prado

In [ ]:
GITHUB_USUARIO = "TU-USUARIO-AQUI"  # <-- cambia esto

%cd /content
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git clone https://github.com/fapi19/Tesis_Spa-Jeb.git 4_aumento_datos/tesis_spa_jeb

## 3. Instalar dependencias (las que F. Prado ya fijó y probó)

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
!pip install -q -r requirements/nmt.txt
# Fix: requirements/nmt.txt fija torch==2.7.1 pero no toca torchvision, que
# Colab trae preinstalado para OTRA version de torch. Eso rompe el import de
# `transformers` (RuntimeError: operator torchvision::nms does not exist).
# No usamos torchvision para nada en este pipeline de texto, pero
# `transformers` lo importa igual — hay que igualar la version.
!pip install -q torchvision==0.22.1
print('Si Colab pide reiniciar el entorno (boton "RESTART SESSION"), hazlo, y luego sigue con la celda 4 directamente (esta celda 3 no hace falta repetirla).')

## 4. Entrenar la configuración campeona (v2.1b LoRA+)

Esto puede tardar 30 min - 2 horas segun la GPU que te toque. Nota el `%cd`
explícito al inicio: si el entorno se reinició en la celda 3, esto lo
corrige solo.

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
%env USE_TF=0
# El tensorflow preinstalado en Colab quedo incompatible con el protobuf que
# instalan los requisitos de F. Prado. No usamos tensorflow para nada aqui
# (solo PyTorch), asi que le decimos a `transformers` que ni lo intente
# cargar -- evita el ImportError en cascada por protobuf/tensorflow.
!python -m scripts.nmt.30_train_lora \
    --variant xl \
    --rank 32 \
    --alpha 64 \
    --loraplus-lr-ratio 16 \
    --output-dir models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl

## 5. Guardar el checkpoint en Drive (¡no te saltes este paso!)

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
!mkdir -p /content/drive/MyDrive/shiwilu_checkpoint
!cp -r models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl /content/drive/MyDrive/shiwilu_checkpoint/

## 6. Evaluar (deberia dar chrF++ promedio cercano a 44.99)

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
%env USE_TF=0
!python -m scripts.nmt.40_evaluate --checkpoint models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl --split test

## 7. Correr la retrotraducción con tu script

Ya con el checkpoint entrenado, corre la técnica completa (parafraseo + traducción a shiwilu + filtros).

In [ ]:
%cd /content/shiwilu-tesis
%env USE_TF=0
!pip install -q sentence-transformers
!python 4_aumento_datos/retrotraduccion.py \
    --checkpoint 4_aumento_datos/tesis_spa_jeb/models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl

## 8. Descargar el resultado a tu computadora

Descarga solo el CSV generado (no el checkpoint completo, pesa varios GB).

In [ ]:
%cd /content/shiwilu-tesis
from google.colab import files
files.download('4_aumento_datos/salidas/retrotraduccion.csv')